In [1]:
import time
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

d:\Ai Data Engineering\RAG_POC\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


True

In [2]:
# 1. LOAD: Read the PDFs
data_folder = "../data/pdf/"
loader = PyPDFDirectoryLoader(data_folder)
raw_documents = loader.load()
print(f"Successfully loaded {len(raw_documents)} total pages.")

Successfully loaded 200 total pages.


In [3]:
# 2. SPLIT: Break pages into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200 
)
chunks = text_splitter.split_documents(raw_documents)
print(f"Created {len(chunks)} chunks.")

Created 333 chunks.


In [4]:

# 2. Setup Embeddings and Path
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2-preview")
persist_directory = "../chroma_db"

In [5]:
# 3. Batch Processing
batch_size = 20  # Smaller batch size for free tier stability
vectorstore = None

print("Starting robust embedding process...")

for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    success = False
    retries = 0
    
    while not success and retries < 5:
        try:
            if i == 0 and vectorstore is None:
                vectorstore = Chroma.from_documents(
                    documents=batch,
                    embedding=embeddings,
                    persist_directory=persist_directory
                )
            else:
                vectorstore.add_documents(documents=batch)
            
            success = True
            print(f"Processed chunks {i} to {min(i + batch_size, len(chunks))}")
            
            # mandatory rest between successful batches
            time.sleep(12) 
            
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                wait_time = (2 ** retries) * 30  # Wait 30s, 60s, 120s...
                print(f"Quota hit. Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
                retries += 1
            else:
                print(f"❌ Unexpected Error: {e}")
                raise e

print(f"\n Success! All chunks stored in {persist_directory}")

Starting robust embedding process...
Processed chunks 0 to 20
Processed chunks 20 to 40
Processed chunks 40 to 60
Processed chunks 60 to 80
Processed chunks 80 to 100
Processed chunks 100 to 120
Processed chunks 120 to 140
Processed chunks 140 to 160
Processed chunks 160 to 180
Processed chunks 180 to 200
Processed chunks 200 to 220
Processed chunks 220 to 240
Processed chunks 240 to 260
Processed chunks 260 to 280
Processed chunks 280 to 300
Processed chunks 300 to 320
Processed chunks 320 to 333

 Success! All chunks stored in ../chroma_db


In [6]:
# 1. Essential Imports
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 2. Initialize the LLM
# Using Gemini 2.5 Flash as your generation engine
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0,
    max_tokens=None,
    timeout=None
)

# 3. Define the SQL-Focused System Prompt
system_prompt = (
    "You are a technical SQL expert. Use the following pieces of retrieved context "
    "from the provided documentation to answer the user's question. "
    "If the answer is not in the context, say you don't know based on the document. "
    "Provide clear SQL code examples where relevant."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# 4. Create the Chain
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("rag_chain is now defined and ready for your SQL queries!")

rag_chain is now defined and ready for your SQL queries!


In [7]:
print("--- SQL Knowledge Bot is Ready ---")
print("Type 'exit' to quit.")

while True:
    user_query = input("\nAsk a question about SQL: ")
    
    if user_query.lower() in ["exit", "quit", "q"]:
        print("Closing Chat. Happy coding!")
        break
        
    # Generate the response using your RAG chain
    response = rag_chain.invoke({"input": user_query})
    
    print(f"\nAI: {response['answer']}")
    
    # See the specific page numbers retrieved
    sources = set([doc.metadata.get('page', 'N/A') for doc in response['context']])
    print(f"(Sources: Page(s) {', '.join(map(str, sources))})")

--- SQL Knowledge Bot is Ready ---
Type 'exit' to quit.

AI: An **INNER JOIN** is a fundamental type of SQL join used to combine records from two or more tables in a database. It is also referred to as an **EQUIJOIN**.

Here's a breakdown of what an INNER JOIN does:

*   **Returns Matching Rows:** An INNER JOIN returns rows only when there is a match in both tables based on the specified join condition (predicate).
*   **Combines Column Values:** It creates a new result table by combining column values from two tables (e.g., `table1` and `table2`).
*   **Predicate-Based Comparison:** The query compares each row of `table1` with each row of `table2` to find all pairs of rows that satisfy the join predicate. When the predicate is satisfied, column values for each matched pair of rows are combined into a single result row.

**Syntax:**

The basic syntax for an INNER JOIN is as follows:

```sql
SELECT table1.column1, table2.column2...
FROM table1
INNER JOIN table2
ON table1.common_filed = 

GoogleGenerativeAIError: Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'EmbedContentRequest.content contains an empty Part.', 'status': 'INVALID_ARGUMENT'}}